# Sparse Autoencoders & Superposition

[Probing](linear-probes.ipynb) requires you to know what to look for. Sparse autoencoders
invert the problem: **find the features without being told what they are.**

They exist because of **superposition** — the observation that networks represent far more
features than they have dimensions, by storing them as *overlapping, non-orthogonal*
directions. That is possible because features are sparse: only a few are active at once,
so their interference is rare enough to tolerate.

Superposition is why individual neurons are usually uninterpretable. A neuron does not
hold "one feature"; it holds a fragment of many. An SAE is an attempt to undo that mixing
and recover the original, sparse, interpretable basis.

Second topic in the [Interpretability](linear-probes.ipynb) track.

## 1. What & Why

The setup. A layer produces activations `h ∈ ℝᵈ`. The hypothesis is that

```
h ≈ Σ aᵢ · fᵢ        with a sparse and the fᵢ a large overcomplete dictionary
```

where there are far more features `fᵢ` than dimensions `d`, and only a handful of the
coefficients `aᵢ` are non-zero for any given input.

An SAE learns that decomposition with an autoencoder that is deliberately **wider than its
input** and **sparsity-penalised**:

```
a = ReLU(W_enc·h + b_enc)          # sparse codes, dimension >> d
ĥ = W_dec·a + b_dec                # reconstruction
L = ‖h − ĥ‖² + λ‖a‖₁
```

The two terms are in direct tension, and that tension is the entire method: perfect
reconstruction is trivial with a dense code, and perfect sparsity reconstructs nothing.

**Why this is worth the trouble:** the features an SAE recovers are often far more
interpretable than neurons, they are found without supervision, and they can be
intervened on — clamp a feature and see what the model does. That last property is what
makes SAEs a causal tool rather than another correlational one.

## 2. Mental Model

**A cocktail party recorded on too few microphones.**

Fifty people are talking; you have ten microphones. Each recording is a mixture of all
fifty voices, so no single microphone corresponds to a person — which is exactly the
neuron-polysemanticity problem.

You can still separate the voices, because of one saving fact: **at any moment only two or
three people are actually speaking.** Sparsity is what makes the underdetermined problem
solvable. If all fifty spoke constantly, ten microphones could not recover them and no
amount of cleverness would help.

That is superposition in one image, and it yields the practical intuitions directly:

- **More features than dimensions is fine if activation is rare.** The interference
  between two features only matters when both are active.
- **The dictionary must be overcomplete.** Ten microphones, fifty output channels — you
  are looking for more things than you have measurements.
- **The sparsity penalty is the assumption that few people speak at once.** Set it too low
  and you get fifty channels of mush; too high and you hear only the loudest speaker.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Superposition** | Representing more features than dimensions, as overlapping non-orthogonal directions. Tolerable when features are sparse. |
| **Polysemantic neuron** | A neuron responding to several unrelated things — the symptom of superposition. |
| **Dictionary / feature directions** | The columns of `W_dec`. Each is one learned feature's direction in activation space. |
| **Expansion factor** | Dictionary size ÷ input dimension. Typically 4×–64×. |
| **L1 coefficient `λ`** | The sparsity/reconstruction dial. The central hyperparameter. |
| **L0** | Average number of active features per input. The sparsity you actually report. |
| **Dead feature** | A dictionary element that never activates. Wasted capacity, and common. |
| **Shrinkage** | L1 biases active coefficients toward zero, so reconstructions are systematically under-scaled. |
| **Feature splitting** | With a larger dictionary, one feature resolves into several finer ones. Not a bug — a resolution choice. |
| **TopK / JumpReLU SAEs** | Variants that fix the sparsity level directly instead of via an L1 penalty, avoiding shrinkage. |
| **Absorption** | One feature quietly absorbing part of another's behaviour, harming interpretability. |

## 4. Setup

NumPy. Synthetic data with a *known* ground-truth dictionary, so recovery can be measured
rather than eyeballed — the standard failure of SAE work is that on real activations there
is no ground truth to check against.

In [1]:
# %pip install numpy

import numpy as np

rng = np.random.default_rng(0)
print("numpy", np.__version__)

numpy 2.5.1


## 5. Worked Examples

### Example 1 — superposition: why sparsity makes the impossible possible

Pack `n_features` directions into `d < n_features` dimensions and measure how well a
single feature can be recovered, as a function of how often features are active.

In [2]:
D_ACT, N_FEATURES = 16, 64

def random_dictionary(d, n, seed=0):
    r = np.random.default_rng(seed)
    F = r.normal(0, 1, (n, d))
    return F / np.linalg.norm(F, axis=1, keepdims=True)

F_true = random_dictionary(D_ACT, N_FEATURES, seed=3)
print(f"{N_FEATURES} features packed into {D_ACT} dimensions "
      f"({N_FEATURES/D_ACT:.0f}x overcomplete)\n")
off = F_true @ F_true.T
np.fill_diagonal(off, 0)
print(f"mean |cosine| between distinct feature directions: {np.abs(off).mean():.3f}")
print("  (0 would be orthogonal -- impossible with more features than dimensions)\n")

# Hold feature 0 ALWAYS active and vary how many others fire alongside it, so the
# signal is constant and only the interference changes.
print(f"{'other features active':>22} {'recovery of feature 0':>22}")
for n_interfere in (0, 1, 2, 4, 9, 24, 49):
    r = np.random.default_rng(7)
    n = 4000
    codes = np.zeros((n, N_FEATURES))
    for i in range(n):
        codes[i, 0] = r.uniform(0.5, 1.5)
        if n_interfere:
            others = r.choice(np.arange(1, N_FEATURES), n_interfere, replace=False)
            codes[i, others] = r.uniform(0.5, 1.5, n_interfere)
    acts = codes @ F_true
    # Read feature 0 back out by projecting onto its own direction.
    corr = float(np.corrcoef(acts @ F_true[0], codes[:, 0])[0, 1])
    print(f"{n_interfere:22d} {corr:22.3f}")

print("\nWith feature 0 alone, a simple projection recovers it perfectly -- despite")
print("there being four times more features than dimensions. As more features fire")
print("alongside it, their non-orthogonal overlap leaks into the readout and recovery")
print("degrades steadily.")
print("\nThat is superposition's bargain: you may store far more features than you have")
print("dimensions, PROVIDED they are rarely active simultaneously. It is also why")
print("individual neurons look polysemantic -- each dimension carries a piece of many")
print("features, and only the sparse combination is meaningful.")

64 features packed into 16 dimensions (4x overcomplete)

mean |cosine| between distinct feature directions: 0.206
  (0 would be orthogonal -- impossible with more features than dimensions)

 other features active  recovery of feature 0
                     0                  1.000
                     1                  0.741
                     2                  0.621
                     4                  0.482
                     9                  0.353
                    24                  0.290
                    49                  0.291

With feature 0 alone, a simple projection recovers it perfectly -- despite
there being four times more features than dimensions. As more features fire
alongside it, their non-orthogonal overlap leaks into the readout and recovery
degrades steadily.

That is superposition's bargain: you may store far more features than you have
dimensions, PROVIDED they are rarely active simultaneously. It is also why
individual neurons look polysemantic 

### Example 2 — train an SAE, and check it against the ground truth

Now recover the dictionary without being told it. Because the true features are known
here, recovery can be measured.

In [3]:
N_ACTIVE = 2       # the true sparsity: 2 of 64 features fire per input

def make_data(n, n_active=N_ACTIVE, seed=0):
    r = np.random.default_rng(seed)
    codes = np.zeros((n, N_FEATURES))
    for i in range(n):
        idx = r.choice(N_FEATURES, n_active, replace=False)
        codes[i, idx] = r.uniform(0.5, 1.5, n_active)
    return codes @ F_true, codes

X, true_codes = make_data(6000, seed=2)

def train_sae(X, n_dict, l1=0.3, steps=4000, lr=0.01, seed=0):
    '''An SAE trained with Adam. Plain gradient descent converges far too slowly here
    to recover the dictionary, which is itself worth knowing.'''
    r = np.random.default_rng(seed)
    W_dec = r.normal(0, 1, (n_dict, X.shape[1]))
    W_dec /= np.linalg.norm(W_dec, axis=1, keepdims=True)
    W_enc = W_dec.T.copy()                     # tied init: a much better starting point
    b_enc = np.zeros(n_dict)
    b1, b2 = 0.9, 0.999
    state = {k: [np.zeros_like(p), np.zeros_like(p)]
             for k, p in (("e", W_enc), ("d", W_dec), ("b", b_enc))}
    for t in range(1, steps + 1):
        a = np.maximum(0, X @ W_enc + b_enc)                 # sparse codes
        err = a @ W_dec - X
        g_dec = a.T @ err / len(X)
        ga = (err @ W_dec.T + l1 * (a > 0)) * (a > 0)        # ReLU gradient
        g_enc = X.T @ ga / len(X)
        g_b = ga.mean(0)
        for key, g, p in (("e", g_enc, W_enc), ("d", g_dec, W_dec), ("b", g_b, b_enc)):
            m, v = state[key]
            m[:] = b1 * m + (1 - b1) * g
            v[:] = b2 * v + (1 - b2) * g ** 2
            p -= lr * (m / (1 - b1 ** t)) / (np.sqrt(v / (1 - b2 ** t)) + 1e-8)
        W_dec /= np.linalg.norm(W_dec, axis=1, keepdims=True) + 1e-8   # unit-norm, standard
    return W_enc, b_enc, W_dec

W_enc, b_enc, W_dec = train_sae(X, n_dict=N_FEATURES * 2, l1=0.3)

a = np.maximum(0, X @ W_enc + b_enc)
recon = a @ W_dec
fvu = float(np.sum((X - recon) ** 2) / np.sum((X - X.mean(0)) ** 2))
l0 = float(np.mean(np.sum(a > 1e-6, axis=1)))

# How many true features found a matching dictionary element?
sim = np.abs(F_true @ W_dec.T)
best = sim.max(axis=1)
print(f"dictionary size      : {W_dec.shape[0]} (2x the true {N_FEATURES})")
print(f"fraction variance unexplained: {fvu:.4f}")
print(f"L0 (active features/input)  : {l0:.2f}   (true sparsity was {N_ACTIVE})")
print(f"\ntrue features recovered at cosine > 0.9: "
      f"{int(np.sum(best > 0.9))}/{N_FEATURES}")
print(f"true features recovered at cosine > 0.7: "
      f"{int(np.sum(best > 0.7))}/{N_FEATURES}")
print(f"mean best-match cosine: {best.mean():.3f}")
print("\nThe SAE was given no information about the true dictionary -- only the mixed")
print("activations -- and recovers a substantial share of the real feature directions.")
print("That is the claim SAEs rest on, demonstrated where it can be checked.")

dictionary size      : 128 (2x the true 64)
fraction variance unexplained: 0.2354
L0 (active features/input)  : 4.81   (true sparsity was 2)

true features recovered at cosine > 0.9: 62/64
true features recovered at cosine > 0.7: 64/64
mean best-match cosine: 0.958

The SAE was given no information about the true dictionary -- only the mixed
activations -- and recovers a substantial share of the real feature directions.
That is the claim SAEs rest on, demonstrated where it can be checked.


### Example 3 — the sparsity/reconstruction frontier

The `λ` sweep is the central design decision, and there is no setting that is best on both
axes. This is the curve every SAE paper reports.

In [4]:
print(f"{'lambda':>8} {'L0':>7} {'FVU':>8} {'dead':>6} {'recovered>0.9':>14} "
      f"{'recovered>0.7':>14}")
for l1 in (0.02, 0.05, 0.1, 0.3, 0.6):
    We, be, Wd = train_sae(X, n_dict=N_FEATURES * 2, l1=l1)
    a = np.maximum(0, X @ We + be)
    fvu = float(np.sum((X - a @ Wd) ** 2) / np.sum((X - X.mean(0)) ** 2))
    l0 = float(np.mean(np.sum(a > 1e-6, axis=1)))
    dead = int(np.sum(a.max(axis=0) <= 1e-6))
    best = np.abs(F_true @ Wd.T).max(axis=1)
    print(f"{l1:8.2f} {l0:7.2f} {fvu:8.4f} {dead:6d} "
          f"{int(np.sum(best > 0.9)):14d} {int(np.sum(best > 0.7)):14d}")

print(f"\n(true sparsity is {N_ACTIVE} active features per input, out of {N_FEATURES})")
print("\nSmall lambda reconstructs almost perfectly and is NOT sparse -- L0 far above")
print("the truth -- and recovers few features cleanly, because a dense code can")
print("reconstruct the data using arbitrary mixtures rather than the real directions.")
print("\nAs lambda rises, reconstruction gets worse and feature recovery gets BETTER,")
print("because sparsity is the constraint that makes the decomposition identifiable at")
print("all. The two things you care about move in opposite directions.")
print("\nThat is the frontier, and it is why an SAE cannot be summarised by one number.")
print("On real activations there is no 'recovered' column to consult -- which is the")
print("central methodological difficulty of the whole field.")

  lambda      L0      FVU   dead  recovered>0.9  recovered>0.7


    0.02   33.18   0.0059      7              1             42


    0.05   27.65   0.0282      2              3             61


    0.10   17.16   0.0730      4             23             62


    0.30    4.81   0.2354      7             62             64


    0.60    2.27   0.4727     25             61             64

(true sparsity is 2 active features per input, out of 64)

Small lambda reconstructs almost perfectly and is NOT sparse -- L0 far above
the truth -- and recovers few features cleanly, because a dense code can
reconstruct the data using arbitrary mixtures rather than the real directions.

As lambda rises, reconstruction gets worse and feature recovery gets BETTER,
because sparsity is the constraint that makes the decomposition identifiable at
all. The two things you care about move in opposite directions.

That is the frontier, and it is why an SAE cannot be summarised by one number.
On real activations there is no 'recovered' column to consult -- which is the
central methodological difficulty of the whole field.


### Example 4 — shrinkage, and why TopK variants exist

The L1 penalty does not only decide *which* features fire; it also systematically shrinks
*how much* they fire. That biases every reconstruction.

In [5]:
W_enc, b_enc, W_dec = train_sae(X, n_dict=N_FEATURES * 2, l1=0.3)
a = np.maximum(0, X @ W_enc + b_enc)

# For features that matched a true one, compare the learned magnitude to the truth.
sim = np.abs(F_true @ W_dec.T)
pairs = [(t, int(np.argmax(sim[t]))) for t in range(N_FEATURES) if sim[t].max() > 0.9]
ratios = []
for t, d in pairs[:40]:
    mask = true_codes[:, t] > 0
    if mask.sum() > 20 and a[mask, d].mean() > 1e-6:
        ratios.append(a[mask, d].mean() / true_codes[mask, t].mean())

print(f"matched features examined: {len(ratios)}")
print(f"mean (learned magnitude / true magnitude): {np.mean(ratios):.3f}")
print(f"  a value below 1.0 means the SAE systematically UNDER-estimates activations\n")

# The mechanism: soft-thresholding at lambda/2.
print("the mechanism, in one line -- L1 solves a soft-thresholding problem:")
for true_val in (0.2, 0.5, 1.0, 2.0):
    for lam in (0.05, 0.2):
        print(f"  true {true_val:4.1f}, lambda {lam:4.2f} -> "
              f"recovered {max(0.0, true_val - lam/2):5.2f}")

print("\nEvery active coefficient is pulled toward zero by a constant, so small")
print("activations are hit hardest in relative terms. The reconstruction is biased low")
print("even when the SAE has identified exactly the right features.")
print("\nThis is what TopK and JumpReLU SAEs are for: fix the number of active features")
print("directly and drop the L1 term, so selection and magnitude stop being coupled.")

matched features examined: 40
mean (learned magnitude / true magnitude): 0.394
  a value below 1.0 means the SAE systematically UNDER-estimates activations

the mechanism, in one line -- L1 solves a soft-thresholding problem:
  true  0.2, lambda 0.05 -> recovered  0.18
  true  0.2, lambda 0.20 -> recovered  0.10
  true  0.5, lambda 0.05 -> recovered  0.47
  true  0.5, lambda 0.20 -> recovered  0.40
  true  1.0, lambda 0.05 -> recovered  0.97
  true  1.0, lambda 0.20 -> recovered  0.90
  true  2.0, lambda 0.05 -> recovered  1.98
  true  2.0, lambda 0.20 -> recovered  1.90

Every active coefficient is pulled toward zero by a constant, so small
activations are hit hardest in relative terms. The reconstruction is biased low
even when the SAE has identified exactly the right features.

This is what TopK and JumpReLU SAEs are for: fix the number of active features
directly and drop the L1 term, so selection and magnitude stop being coupled.


## 6. Gotchas & Pitfalls

- **Reading interpretability into a single number.** An SAE with low FVU and low L0 may
  still have uninterpretable features. Reconstruction quality is necessary, not
  sufficient — someone has to look at what the features fire on.
- **Comparing SAEs at one operating point.** Example 3: report the whole frontier.
- **Ignoring dead features.** A large fraction of the dictionary commonly never fires,
  wasting capacity. Resampling or auxiliary losses target this.
- **Forgetting shrinkage.** Example 4. If you use SAE activations quantitatively, they are
  biased low.
- **Assuming feature splitting is a defect.** A bigger dictionary resolving one feature
  into three is a change of resolution, not an error — but it does mean feature counts are
  not comparable across dictionary sizes.
- **Believing the dictionary is unique.** Different seeds give different dictionaries.
  Features that appear across seeds are more trustworthy than any single run's.
- **Interpreting features from top-activating examples only.** That is a biased sample and
  invites confirmation bias. Look at the whole activation distribution, and test the
  hypothesis causally.
- **Assuming the model uses the feature.** Same limitation as [probing](linear-probes.ipynb):
  a direction found by an SAE is a decomposition of the activations, not proof that
  downstream computation reads it. Intervene to check.
- **Expecting recovery on real activations to look like Example 2.** There is no ground
  truth there, and the true feature count is unknown.

## 7. When to Use vs Alternatives

| Question | Tool |
|---|---|
| What features live in this layer, with no label set? | **SAE** |
| Is *this specific* property present? | [**Linear probe**](linear-probes.ipynb) — far cheaper when you know what you want |
| Does the model use this feature? | Clamp/ablate it and measure the output — causal |
| Which components produced this behaviour? | Activation patching / circuit analysis |
| Fixed sparsity without shrinkage | **TopK** or **JumpReLU** SAEs |
| Steering behaviour | [**Activation steering**](activation-steering.ipynb), which can use SAE features as directions |

**The honest position.** SAEs are the most promising unsupervised interpretability method
available and remain unproven in an important sense: on real activations there is no
ground truth, so "did it find the right features" is answered by human judgement of
whether the features look interpretable. That is a weak standard, and recent work has
found real problems — feature absorption, sensitivity to dictionary size, and gaps between
SAE features and the directions that actually matter causally for behaviour.

They are worth learning because they are the main tool for the unsupervised question, and
because the superposition framing in Example 1 is genuinely explanatory regardless of
whether SAEs turn out to be the right way to undo it. Treat their output as hypotheses to
be tested causally, not as a readout of what the model is thinking.

## 8. Resources

- [Toy Models of Superposition](https://transformer-circuits.pub/2022/toy_model/index.html) — Elhage et al.; Example 1's phenomenon, explored far more thoroughly.
- [Towards Monosemanticity: Decomposing Language Models With Dictionary Learning](https://transformer-circuits.pub/2023/monosemantic-features/index.html) — the paper that made SAEs a standard tool.
- [Scaling Monosemanticity](https://transformer-circuits.pub/2024/scaling-monosemanticity/) — SAEs on a production model, including feature steering.
- [Scaling and evaluating sparse autoencoders](https://arxiv.org/abs/2406.04093) — TopK SAEs and the shrinkage problem of Example 4.
- [Improving Dictionary Learning with Gated Sparse Autoencoders](https://arxiv.org/abs/2404.16014) — decoupling selection from magnitude.
- [Sparse Autoencoders Find Highly Interpretable Features in Language Models](https://arxiv.org/abs/2309.08600) — an independent replication with evaluation methodology.
- [Open Problems in Mechanistic Interpretability](https://arxiv.org/abs/2501.16496) — a current, candid account of what SAEs have and have not delivered.